# Session 12 Solutions: Cleaning Messy Data

Data Science with Python, Level 1, Stage 2: Working with Data

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
survey = pd.read_csv(DATA_URL + "messy_survey.csv")

## In-Class Exercises

### In-Class Exercise 1: Count the Problems

In [2]:
raw = pd.read_csv(DATA_URL + "messy_survey.csv")
print(len(raw), raw.duplicated().sum())
print(raw.isna().sum())
print(raw["city"].nunique(dropna=False))

126 6
respondent        0
name              0
age               0
city              4
monthly_income    9
hours_online      6
satisfaction      3
signup_date       0
dtype: int64
9


### In-Class Exercise 2: Clean Names

In [3]:
names = pd.Series(["  lina haddad", "OMAR SALEH ", "sara  nassar"])
clean = names.str.split().str.join(" ").str.title()
print(clean.tolist())

['Lina Haddad', 'Omar Saleh', 'Sara Nassar']


### In-Class Exercise 3: Prices as Text

In [4]:
prices = pd.Series(["12.5", "8", "$15", "n/a", "7,5"])
fixed = prices.str.replace("$", "").str.replace(",", ".")
nums = pd.to_numeric(fixed, errors="coerce")
print(nums.tolist(), nums.isna().sum())

[12.5, 8.0, 15.0, nan, 7.5] 1


### In-Class Exercise 4: Fill Smartly

In [5]:
d = pd.DataFrame({"group": ["A", "A", "B", "B", "B"], "score": [10, None, 30, 34, None]})
d["score"] = d["score"].fillna(d.groupby("group")["score"].transform("median"))
print(d)

  group  score
0     A   10.0
1     A   10.0
2     B   30.0
3     B   34.0
4     B   32.0


## Part A: Basic

### Exercise 1: Spot the Problems

In [6]:
d = pd.DataFrame({"name": [" lina", "OMAR ", "Sara", " lina"],
                  "score": ["88", "75", "ninety", "88"]})
# Problems: a duplicate row, spaces and capitals in names, a score written as a word.
d = d.drop_duplicates()
d["name"] = d["name"].str.strip().str.title()
d["score"] = pd.to_numeric(d["score"].replace({"ninety": "90"}), errors="coerce")
print(d)
print(d.dtypes)

   name  score
0  Lina     88
1  Omar     75
2  Sara     90
name       str
score    int64
dtype: object


### Exercise 2: Missing Values Report

In [7]:
def missing_report(df):
    counts = df.isna().sum()
    report = pd.DataFrame({"missing": counts, "percent": (counts / len(df) * 100).round(1)})
    return report.sort_values("missing", ascending=False)

missing_report(pd.read_csv(DATA_URL + "messy_survey.csv"))

,missing,percent
monthly_income,9,7.1
hours_online,6,4.8
city,4,3.2
satisfaction,3,2.4
age,0,0.0
name,0,0.0
respondent,0,0.0
signup_date,0,0.0


## Part B: Intermediate

### Exercise 3: Standardise Categories

`title()` turns `ML` into `Ml`, so the dictionary fixes it back.

In [8]:
jobs = pd.Series(["Data Analyst", "data analyst ", "Analyst, Data", "DATA ANALYST", "Data Scientist", "data scientist", "ML Engineer"])
clean = jobs.str.strip().str.title().replace({"Analyst, Data": "Data Analyst", "Ml Engineer": "ML Engineer"})
print(clean.value_counts())

Data Analyst      4
Data Scientist    2
ML Engineer       1
Name: count, dtype: int64


### Exercise 4: Clean Dates

In [9]:
dates = pd.Series(["2025-03-14", "14/03/2025", "March 15, 2025", "16.03.2025"])
parsed = pd.to_datetime(dates, format="mixed", dayfirst=True)
print(parsed.dt.strftime("%Y-%m-%d %A").tolist())

['2025-03-14 Friday', '2025-03-14 Friday', '2025-03-15 Saturday', '2025-03-16 Sunday']


## Part C: Challenge

### Exercise 5: A Reusable Cleaning Function

In [10]:
def clean_survey(path):
    s = pd.read_csv(path).drop_duplicates()
    s["city"] = s["city"].str.strip().str.title().fillna("Unknown")
    age = s["age"].astype(str).str.strip().str.replace("years", "").str.replace("yrs", "").replace({"thirty": "30"})
    s["age"] = pd.to_numeric(age, errors="coerce")
    s.loc[~s["age"].between(16, 100), "age"] = np.nan
    s["age"] = s["age"].fillna(s["age"].median())
    s["satisfaction"] = pd.to_numeric(s["satisfaction"].replace({"five": "5"}), errors="coerce")
    s = s.dropna(subset=["satisfaction"])
    s["signup_date"] = pd.to_datetime(s["signup_date"])
    s.loc[s["hours_online"] > 24, "hours_online"] = np.nan
    for col in ["hours_online", "monthly_income"]:
        s[col] = s[col].fillna(s[col].median())
    return s

clean = clean_survey(DATA_URL + "messy_survey.csv")
assert clean.isna().sum().sum() == 0
assert clean.duplicated().sum() == 0
for city in sorted(clean["city"].unique()):
    print(city, round(clean.loc[clean["city"] == city, "satisfaction"].mean(), 2))

Aleppo 3.82
Damascus 3.66
Homs 4.0
Latakia 3.78
Unknown 3.5


## Part D: Cumulative Project: Cafe Sales Analysis

### Exercise 6: Cafe Data Quality Check

In [11]:
df = pd.read_csv(DATA_URL + "cafe_sales.csv")
print("Duplicate rows:", df.duplicated().sum(), "duplicate IDs:", df["order_id"].duplicated().sum())
print(df.isna().sum())
print("Bad quantities:", (df["quantity"] < 1).sum(), "bad prices:", (df["unit_price"] < 0).sum())
inconsistent = [item for item in df["item"].unique() if df.loc[df["item"] == item, "unit_price"].nunique() != 1]
print("Items with more than one price:", inconsistent)
print("Hours in range:", df["hour"].between(7, 20).all())
# The cafe data passes every check, so it is clean enough to analyse.

Duplicate rows: 0 duplicate IDs: 0
order_id      0
date          0
hour          0
branch        0
item          0
category      0
unit_price    0
quantity      0
payment       0
dtype: int64
Bad quantities: 0 bad prices: 0
Items with more than one price: []
Hours in range: True


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | `isna` marks missing values; `sum` counts them per column. |
| 2 | C | With `coerce`, values that cannot be converted become missing. |
| 3 | B | A few very high incomes pull the mean up but not the median. |
| 4 | B | `subset` limits the check to the listed columns. |
| 5 | C | `strip` removes the spaces and `title` capitalises the first letter. |